# Bank Customer Churn Dataset - Data Preprocessing

This notebook adapts the preprocessing workflow to your **Bank Customer Churn Prediction** project using `Churn_Modelling.csv`.

**Goal:** clean the dataset, remove irrelevant columns, handle duplicates, inspect outliers, encode categorical features, scale numeric features, and save a processed dataset for model training.


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style("whitegrid")

In [ ]:
from pathlib import Path

BASE_DIR = Path("..").resolve()
FIGURES_DIR = BASE_DIR / "outputs" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR

## Load Dataset

In [ ]:
DATA_PATH = BASE_DIR / "data" / "raw" / "Churn_Modelling.csv"

df = pd.read_csv(DATA_PATH)
df.head()


## Basic Information

In [ ]:
print("Shape of dataset:", df.shape)
display(df.head())
display(df.describe(include='all'))
df.info()

## Check Missing Values

In [ ]:
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

print("Missing values per column:")
display(missing_values)

print("Missing percentage per column:")
display(missing_percent)

## Check Duplicate Rows

This dataset usually does not contain duplicate rows, but we verify it before moving forward.


In [ ]:
print("Duplicate rows before removal:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after duplicate removal:", df.shape)
print("Duplicate rows after removal:", df.duplicated().sum())

## Drop Irrelevant Columns

The following columns are not useful for prediction:
- `RowNumber` -> row index only
- `CustomerId` -> unique identifier
- `Surname` -> customer name, not a stable predictive feature for this project


In [ ]:
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df = df.drop(columns=columns_to_drop)
df.head()

## Identify Numerical and Categorical Columns

In [ ]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)

## Target Variable Distribution

In [ ]:
df['Exited'].value_counts().sort_index().plot(kind='bar')
plt.title('Target Distribution - Exited')
plt.xlabel('Exited')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "class_distribution.png")

plt.show()

print(df['Exited'].value_counts())
print(df['Exited'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Geography', hue='Exited')
plt.title('Churn by Geography')
plt.xlabel('Geography')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "churn_by_geography.png")
plt.show()



In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(data=df, x='Gender', hue='Exited')
plt.title('Churn by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "churn_by_gender.png")
plt.show()



In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Age', kde=True, bins=30)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "age_distribution.png")
plt.show()



In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Balance', kde=True, bins=30)
plt.title('Balance Distribution')
plt.xlabel('Balance')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "balance_distribution.png")
plt.show()



In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='Exited', y='Age')
plt.title('Age vs Churn')
plt.xlabel('Exited')
plt.ylabel('Age')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "age_vs_churn_boxplot.png")
plt.show()



In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='Exited', y='Balance')
plt.title('Balance vs Churn')
plt.xlabel('Exited')
plt.ylabel('Balance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "balance_vs_churn_boxplot.png")
plt.show()



In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='Exited', y='CreditScore')
plt.title('Credit Score vs Churn')
plt.xlabel('Exited')
plt.ylabel('Credit Score')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "creditscore_vs_churn_boxplot.png")
plt.show()



## Inspect Categorical Columns

In [ ]:
for col in categorical_columns:
    print(f"\nColumn: {col}")
    display(df[col].value_counts())

## Visualize Outliers with Boxplots

We inspect outliers in numeric features.  
For churn prediction, columns such as `Age`, `CreditScore`, and `NumOfProducts` may show extreme values.


In [ ]:
feature_columns_for_boxplot = [col for col in numerical_columns if col != 'Exited']

plt.figure(figsize=(14, 10))
for i, col in enumerate(feature_columns_for_boxplot, 1):
    plt.subplot((len(feature_columns_for_boxplot) // 3) + 1, 3, i)
    sns.boxplot(x=df[col])
    plt.title(col)

plt.tight_layout()

plt.savefig(FIGURES_DIR / "numerical_boxplots.png")

plt.show()

## Handle Outliers Using IQR Capping

We cap extreme values instead of removing rows, so we do not lose useful customer records.


In [ ]:
def outlier_handler_iqr(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_count = ((dataframe[column] < lower_limit) | (dataframe[column] > upper_limit)).sum()
    dataframe[column] = dataframe[column].clip(lower=lower_limit, upper=upper_limit)

    return outlier_count, lower_limit, upper_limit

outlier_summary = []

for col in ['CreditScore', 'Age', 'NumOfProducts']:
    count, lower, upper = outlier_handler_iqr(df, col)
    outlier_summary.append([col, count, lower, upper])

outlier_summary_df = pd.DataFrame(outlier_summary, columns=['Column', 'Outlier Count', 'Lower Limit', 'Upper Limit'])
outlier_summary_df

In [ ]:
plt.figure(figsize=(14, 6))
for i, col in enumerate(['CreditScore', 'Age', 'NumOfProducts'], 1):
    plt.subplot(1, 3, i)
    sns.boxplot(x=df[col])
    plt.title(f'{col} After Capping')

plt.tight_layout()

plt.savefig(FIGURES_DIR / "capped_boxplots.png")

plt.show()

## Encode Categorical Features

In [ ]:
# Gender -> Binary Encoding
gender_encoder = LabelEncoder()
df['Gender'] = gender_encoder.fit_transform(df['Gender'])   # Female/Male -> 0/1

# Geography -> One-Hot Encoding
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

df.head()

## Correlation Analysis

This helps identify which features are positively or negatively related to customer churn.


In [ ]:
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()

plt.savefig(FIGURES_DIR / "correlation_heatmap.png")

plt.show()

In [ ]:
target_corr = correlation_matrix['Exited'].drop('Exited').sort_values(ascending=False)
display(target_corr)

plt.figure(figsize=(10, 5))
target_corr.plot(kind='bar')
plt.title('Feature Correlation with Exited')
plt.ylabel('Correlation')
plt.tight_layout()

plt.savefig(FIGURES_DIR / "feature_correlation_with_exited.png")

plt.show()

## Scale Numerical Features

Scaling is useful especially for:
- Logistic Regression
- SVM

Tree-based models such as Decision Tree and Random Forest do not strictly require scaling, but keeping a scaled version is convenient for your project.


In [ ]:
columns_to_scale = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

scaler = StandardScaler()
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

df.head()

## Final Processed Dataset Info

In [ ]:
print("Final shape:", df.shape)
display(df.head())
display(df.describe())

## Save Processed Dataset

In [ ]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "churn_processed.csv"

df.to_csv(output_path, index=False)
print(f'Processed dataset saved to: {output_path}')

## Ready for Model Training

You can now use this processed dataset for:
- Logistic Regression
- Decision Tree
- SVM
- Random Forest
